# quantbench_ensemble_current_walkforward_v1

Purpose: refresh the most promising QuantBench ensemble family on current SOXL qlib data, then evaluate it against the current retention-style requirements: walk-forward MaxDD < 20%, YTD return above the SOXLBacktest reference baseline, and YTD return aligned to 95% of the current best replay model.

Governance:
- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparisons `quantbench_paper_soxl_v2`, `model_retention_walkforward_gate_v1`, `cap10_three_fold_checkpoint_metric_replay_v2`, and `soxlbacktest_teacher_risk_budget_feasibility_v1`.
- Role: `diagnose` / training-target selection, not actor promotion.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`, `ytd_2026`, and `latest_2025_2026` through `safe_backtest_end`.
- Changed layer: `model_selection`, `training_target`, `quantbench_path_ensemble`.
- External reference: QuantBench-style supervised/ensemble model matrix plus repo-owned SOXL/SOXX/VIX feature packs.
- Hypothesis: the prior eligible `quantbench_ensemble_sequence_fragile_panic_budget_v1` may preserve <20% drawdown while improving return enough to serve as the next BC/PPO target.
- Acceptance: a refreshed path must pass all required walk-forward drawdown gates and YTD return gates before it is marked target-ready. It is still not a retained actor.
- Failure mode: if YTD/bull capture or any walk-forward MaxDD fails, use the failure columns to choose a narrower target revision rather than promoting or training blindly.
- Comparison-table row key: `quantbench_ensemble_current_walkforward_v1`.

Historical context:
- Closest success: `quantbench_paper_soxl_v2` found one eligible ensemble under the QuantBench proxy gate, but its daily path ended at 2026-03-31.
- Closest failure: `cap10_three_fold_checkpoint_metric_replay_v2` failed `wf_2020`, and `soxlbacktest_teacher_risk_budget_feasibility_v1` found 0/15156 teacher-risk-budget candidates.
- Non-duplicate reason: this notebook refreshes a supervised ensemble family to the current qlib safe end and evaluates retention-style YTD/walk-forward gates, rather than reusing stale proxy rows or another SOXLBacktest teacher cap screen.


In [1]:
from __future__ import annotations

import json
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError("repo root not found")


ROOT = find_repo_root()
CARD = ROOT / "runtime" / "model_selection" / "card" / "quantbench_ensemble_current_walkforward_v1"
CARD.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import torch
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

from src.soxl_quant.app.model_retention_gate import load_reference_baseline, load_ytd_comparison
from src.soxl_quant.app.quantbench_ensemble import (
    build_quantbench_supervised_ensemble_report,
    default_quantbench_path_selector_configs,
)
from src.soxl_quant.app.quantbench_screen import QuantBenchGate
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = "quantbench_ensemble_current_walkforward_v1"
TARGET_MAX_DD = 20.0
BEST_RETURN_RATIO = 0.95
MODEL_NAMES = ("random_forest_clf", "sequence_lag_ridge_mse")
CONFIG_NAMES = (
    "quantbench_ensemble_sequence_fragile_panic_budget_v1",
    "quantbench_ensemble_sequence_fragile_preshock_v1",
    "quantbench_ensemble_sequence_tapered_reentry_v1",
)


def write_json(path: Path, payload: object) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, allow_nan=True) + "\n", encoding="utf-8")


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if not np.isfinite(value):
            return None
        return float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        try:
            return str(value.relative_to(ROOT))
        except ValueError:
            return str(value)
    return value

print(f"root={ROOT}")
print(f"card={CARD.relative_to(ROOT)}")


root=/Users/rziiiii/dev/workspace/.worktrees/wf2020-bull-diagnostic
card=runtime/model_selection/card/quantbench_ensemble_current_walkforward_v1


In [2]:
freshness_path = ROOT / "dashboard_data" / "model_replay_freshness.json"
freshness = json.loads(freshness_path.read_text(encoding="utf-8"))
safe_backtest_end = str(freshness["safe_backtest_end"])
latest_data_date = str(freshness["latest_data_date"])
reference = load_reference_baseline(ROOT)
ytd_comparison = load_ytd_comparison(ROOT)
ytd_return_col = "model_return_pct" if "model_return_pct" in ytd_comparison.columns else "current_model_return_pct"
current_best_return_pct = float(ytd_comparison[ytd_return_col].max())
required_best_return_pct = current_best_return_pct * BEST_RETURN_RATIO

feature_packs = (
    "vix_term",
    "academic_signals",
    "macro_signals",
    "rulebase_signals",
    "transition_signals",
    "soxx_signals",
    "soxx_spread_oscillation",
)

windows = {
    "wf_2020": ("2020-01-01", "2020-12-31"),
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
    # Keep QuantBench's expected slice names too, so its native screen is not empty.
    "test_2020": ("2020-01-01", "2020-12-31"),
    "test_2022": ("2022-01-01", "2022-12-31"),
    "test_2023_2024": ("2023-01-01", "2024-12-31"),
    "latest_2025_2026": ("2025-01-01", safe_backtest_end),
    "ytd_2026": ("2026-01-01", safe_backtest_end),
}

context = {
    "safe_backtest_end": safe_backtest_end,
    "latest_data_date": latest_data_date,
    "reference_profile": "soxl_backtest_policy_reimplementation_v1",
    "reference_ytd_return_pct": reference.return_pct,
    "reference_ytd_max_drawdown_pct": reference.max_drawdown_pct,
    "current_best_return_pct": current_best_return_pct,
    "best_return_ratio": BEST_RETURN_RATIO,
    "required_best_return_pct": required_best_return_pct,
    "target_max_dd_pct": TARGET_MAX_DD,
    "feature_packs": feature_packs,
    "model_names": MODEL_NAMES,
    "config_names": CONFIG_NAMES,
    "windows": windows,
    "source_cards": {
        "stale_quantbench": "runtime/model_selection/card/quantbench_paper_soxl_v2",
        "retention_gate": "runtime/model_selection/card/model_retention_walkforward_gate_v1",
        "checkpoint_metric_replay": "runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2",
        "teacher_feasibility": "runtime/model_selection/card/soxlbacktest_teacher_risk_budget_feasibility_v1",
    },
}
write_json(CARD / "context.json", json_safe(context))
print(json.dumps(json_safe(context), indent=2, allow_nan=False))


{
  "safe_backtest_end": "2026-06-11",
  "latest_data_date": "2026-06-12",
  "reference_profile": "soxl_backtest_policy_reimplementation_v1",
  "reference_ytd_return_pct": 132.07797795135497,
  "reference_ytd_max_drawdown_pct": 13.55408854280352,
  "current_best_return_pct": 224.36153876475817,
  "best_return_ratio": 0.95,
  "required_best_return_pct": 213.14346182652025,
  "target_max_dd_pct": 20.0,
  "feature_packs": [
    "vix_term",
    "academic_signals",
    "macro_signals",
    "rulebase_signals",
    "transition_signals",
    "soxx_signals",
    "soxx_spread_oscillation"
  ],
  "model_names": [
    "random_forest_clf",
    "sequence_lag_ridge_mse"
  ],
  "config_names": [
    "quantbench_ensemble_sequence_fragile_panic_budget_v1",
    "quantbench_ensemble_sequence_fragile_preshock_v1",
    "quantbench_ensemble_sequence_tapered_reentry_v1"
  ],
  "windows": {
    "wf_2020": [
      "2020-01-01",
      "2020-12-31"
    ],
    "wf_2022": [
      "2022-01-01",
      "2022-12-31"
  

In [3]:
init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=feature_packs, profile_name=EXPERIMENT)
features = dataset.handler.feature_frame
close = dataset.handler.price_frame["price"]

if isinstance(features.index, pd.MultiIndex):
    feature_dates = pd.DatetimeIndex(pd.to_datetime(features.index.get_level_values("datetime")))
else:
    feature_dates = pd.DatetimeIndex(pd.to_datetime(features.index))
if isinstance(close.index, pd.MultiIndex):
    close_dates = pd.DatetimeIndex(pd.to_datetime(close.index.get_level_values("datetime")))
else:
    close_dates = pd.DatetimeIndex(pd.to_datetime(close.index))

data_context = {
    "feature_rows": int(len(features)),
    "feature_count": int(features.shape[1]),
    "feature_start": str(feature_dates.min().date()),
    "feature_end": str(feature_dates.max().date()),
    "close_start": str(close_dates.min().date()),
    "close_end": str(close_dates.max().date()),
    "safe_backtest_end": safe_backtest_end,
}
write_json(CARD / "data_context.json", data_context)
print(json.dumps(data_context, indent=2))


[13113:MainThread](2026-06-13 21:13:32,163) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[13113:MainThread](2026-06-13 21:13:32,674) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[13113:MainThread](2026-06-13 21:13:32,675) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/wf2020-bull-diagnostic/data/qlib/soxl')}


[13113:MainThread](2026-06-13 21:13:34,474) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[13113:MainThread](2026-06-13 21:13:34,475) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[13113:MainThread](2026-06-13 21:13:34,475) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


{
  "feature_rows": 3835,
  "feature_count": 113,
  "feature_start": "2011-03-11",
  "feature_end": "2026-06-10",
  "close_start": "2011-03-11",
  "close_end": "2026-06-10",
  "safe_backtest_end": "2026-06-11"
}


In [4]:
all_configs = default_quantbench_path_selector_configs(profile_prefix="quantbench_supervised")
selected_configs = [config for config in all_configs if config.name in CONFIG_NAMES]
missing_configs = sorted(set(CONFIG_NAMES) - {config.name for config in selected_configs})
if missing_configs:
    raise RuntimeError(f"missing QuantBench configs: {missing_configs}")
config_rows = [asdict(config) for config in selected_configs]
pd.DataFrame(config_rows).to_csv(CARD / "ensemble_config_subset.csv", index=False)
print(pd.DataFrame({"name": [config.name for config in selected_configs]}))


                                                name
0    quantbench_ensemble_sequence_tapered_reentry_v1
1   quantbench_ensemble_sequence_fragile_preshock_v1
2  quantbench_ensemble_sequence_fragile_panic_bud...


In [5]:
gate = QuantBenchGate(max_drawdown_pct=TARGET_MAX_DD)
ensemble_report = build_quantbench_supervised_ensemble_report(
    features,
    close,
    card_dir=CARD,
    root=ROOT,
    horizon=20,
    model_names=MODEL_NAMES,
    feature_packs=feature_packs,
    gate=gate,
    windows=windows,
    test_end=safe_backtest_end,
    ensemble_configs=selected_configs,
    rolling_enabled=False,
    hazard_enabled=True,
)
print(json.dumps({
    "decision": ensemble_report["decision"],
    "records": ensemble_report["screen"]["records"],
    "eligible_profiles": ensemble_report["screen"]["eligible_profiles"],
    "best_profile": ensemble_report["screen"]["best_rows"][0]["profile"] if ensemble_report["screen"]["best_rows"] else None,
    "data_end": ensemble_report["data"].get("end"),
    "model_names": ensemble_report["data"].get("model_names"),
}, indent=2))


{
  "decision": "eligible_ensemble_candidates_found",
  "records": 3,
  "eligible_profiles": [
    "quantbench_ensemble_sequence_fragile_panic_budget_v1"
  ],
  "best_profile": "quantbench_ensemble_sequence_fragile_panic_budget_v1",
  "data_end": "2026-06-10",
  "model_names": [
    "random_forest_clf",
    "sequence_lag_ridge_mse"
  ]
}


In [6]:
payloads = json.loads((CARD / "ensemble_payloads.json").read_text(encoding="utf-8"))
rows = []
for payload in payloads:
    profile = str(payload["profile"])
    proxy = payload.get("test_proxy", {})
    slices = payload.get("slices", {})
    ytd = slices.get("ytd_2026", {})
    failures = []
    required_wf = ("wf_2020", "wf_2022", "wf_2023_2024")
    for window in required_wf:
        metrics = slices.get(window)
        if not metrics:
            failures.append(f"missing_wf_window:{window}")
            continue
        if float(metrics.get("max_dd", float("inf"))) >= TARGET_MAX_DD:
            failures.append(f"wf_max_drawdown_ge_20:{window}")
    if not ytd:
        failures.append("missing_ytd_2026")
        ytd_return = float("nan")
        ytd_max_dd = float("nan")
    else:
        ytd_return = float(ytd.get("return", float("nan")))
        ytd_max_dd = float(ytd.get("max_dd", float("nan")))
        if ytd_return < reference.return_pct:
            failures.append("ytd_return_below_soxlbacktest_baseline")
        if ytd_return < required_best_return_pct:
            failures.append("ytd_return_below_best_alignment")
        if ytd_max_dd >= TARGET_MAX_DD:
            failures.append("ytd_max_drawdown_ge_20")
    row = {
        "profile": profile,
        "retention_style_gate": len(failures) == 0,
        "target_ready": len(failures) == 0,
        "failures": ";".join(failures),
        "proxy_return_pct": float(proxy.get("return", float("nan"))),
        "proxy_max_dd_pct": float(proxy.get("max_dd", float("nan"))),
        "ytd_return_pct": ytd_return,
        "ytd_max_dd_pct": ytd_max_dd,
        "required_best_return_pct": required_best_return_pct,
        "required_soxlbacktest_return_pct": reference.return_pct,
    }
    for window in ["wf_2020", "wf_2022", "wf_2023_2024", "latest_2025_2026"]:
        metrics = slices.get(window, {})
        row[f"{window}_return_pct"] = float(metrics.get("return", float("nan")))
        row[f"{window}_buyhold_return_pct"] = float(metrics.get("buy_and_hold_return", float("nan")))
        row[f"{window}_relative_return_pct"] = float(metrics.get("relative_return", float("nan")))
        row[f"{window}_max_dd_pct"] = float(metrics.get("max_dd", float("nan")))
        row[f"{window}_avg_pos"] = float(metrics.get("avg_pos", float("nan")))
    rows.append(row)

retention_gate = pd.DataFrame(rows).sort_values(
    ["retention_style_gate", "ytd_return_pct", "proxy_max_dd_pct"],
    ascending=[False, False, True],
).reset_index(drop=True)
retention_gate.to_csv(CARD / "retention_style_gate.csv", index=False)
display(retention_gate)


,profile,retention_style_gate,target_ready,failures,proxy_return_pct,proxy_max_dd_pct,ytd_return_pct,ytd_max_dd_pct,required_best_return_pct,required_soxlbacktest_return_pct,...,wf_2023_2024_return_pct,wf_2023_2024_buyhold_return_pct,wf_2023_2024_relative_return_pct,wf_2023_2024_max_dd_pct,wf_2023_2024_avg_pos,latest_2025_2026_return_pct,latest_2025_2026_buyhold_return_pct,latest_2025_2026_relative_return_pct,latest_2025_2026_max_dd_pct,latest_2025_2026_avg_pos
0,quantbench_ensemble_sequence_fragile_panic_bud...,False,False,ytd_return_below_soxlbacktest_baseline;ytd_ret...,316.604032,19.742491,15.501901,12.299387,213.143462,132.077978,...,143.747559,191.773509,-48.025950,15.205250,0.234938,43.869051,552.873124,-509.004073,12.299387,0.157752
1,quantbench_ensemble_sequence_fragile_preshock_v1,False,False,wf_max_drawdown_ge_20:wf_2020;ytd_return_below...,333.154961,20.554343,15.408210,12.238721,213.143462,132.077978,...,142.297470,191.773509,-49.476039,15.199416,0.235309,47.668172,552.873124,-505.204952,12.238721,0.169751
2,quantbench_ensemble_sequence_tapered_reentry_v1,False,False,wf_max_drawdown_ge_20:wf_2020;ytd_return_below...,325.581049,20.554263,15.407420,12.238519,213.143462,132.077978,...,145.113323,191.773509,-46.660186,15.199434,0.240941,46.580763,552.873124,-506.292361,12.238519,0.169322


In [7]:
selected = retention_gate.iloc[0].to_dict() if not retention_gate.empty else {}
all_gate_count = int(retention_gate["retention_style_gate"].sum()) if not retention_gate.empty else 0
eligible_rows = retention_gate[retention_gate["retention_style_gate"]].copy() if not retention_gate.empty else pd.DataFrame()

if all_gate_count > 0:
    decision = "quantbench_current_walkforward_target_ready_no_actor_promotion"
    next_mechanism = "distill_quantbench_path_to_fold_local_bc_or_actor_policy_then_run_retention_gate"
else:
    decision = "quantbench_current_walkforward_not_target_ready_no_promotion"
    next_mechanism = "use_failure_columns_to_tune_bull_release_or_drawdown_budget_before_actor_training"

selected_profile = str(selected.get("profile", ""))
selected_daily = CARD / f"{selected_profile}_soxl_daily.csv"
selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "next_mechanism": next_mechanism,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": [
        "quantbench_paper_soxl_v2",
        "model_retention_walkforward_gate_v1",
        "cap10_three_fold_checkpoint_metric_replay_v2",
        "soxlbacktest_teacher_risk_budget_feasibility_v1",
    ],
    "reference_baseline": "soxlbacktest_qlib_buysell_reference",
    "role": "diagnose_training_target",
    "changed_layer": ["model_selection", "training_target", "quantbench_path_ensemble"],
    "target_max_dd_pct": TARGET_MAX_DD,
    "best_return_ratio": BEST_RETURN_RATIO,
    "required_best_return_pct": required_best_return_pct,
    "required_soxlbacktest_return_pct": reference.return_pct,
    "safe_backtest_end": safe_backtest_end,
    "latest_data_date": latest_data_date,
    "model_names": list(MODEL_NAMES),
    "config_names": list(CONFIG_NAMES),
    "all_gate_count": all_gate_count,
    "candidate_count": int(len(retention_gate)),
    "selected_profile": selected_profile,
    "selected_retention_style_gate": bool(selected.get("retention_style_gate", False)),
    "selected_failures": str(selected.get("failures", "")),
    "selected_proxy_return_pct": float(selected.get("proxy_return_pct", float("nan"))),
    "selected_proxy_max_dd_pct": float(selected.get("proxy_max_dd_pct", float("nan"))),
    "selected_ytd_return_pct": float(selected.get("ytd_return_pct", float("nan"))),
    "selected_ytd_max_dd_pct": float(selected.get("ytd_max_dd_pct", float("nan"))),
    "selected_wf_max_dd_pct": {
        window: float(selected.get(f"{window}_max_dd_pct", float("nan")))
        for window in ("wf_2020", "wf_2022", "wf_2023_2024")
    },
    "selected_wf_return_pct": {
        window: float(selected.get(f"{window}_return_pct", float("nan")))
        for window in ("wf_2020", "wf_2022", "wf_2023_2024")
    },
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": bool(all_gate_count > 0),
    "selected_daily_path": str(selected_daily.relative_to(ROOT)) if selected_daily.exists() else "",
    "promotion_blocker": "No actor bundle is trained or retained by this notebook; a target-ready path still must be distilled into an actor and replayed through the retention gate.",
    "sidecars": [
        "context.json",
        "data_context.json",
        "ensemble_config_subset.csv",
        "ensemble_candidate_screen.csv",
        "ensemble_payloads.json",
        "ensemble_path_library.csv",
        "ensemble_objective_metrics.json",
        "ensemble_hazard_metrics.json",
        "retention_style_gate.csv",
        "selection.json",
        "summary.json",
        "conclusion.md",
    ],
}
summary = {
    **selection,
    "conclusion": [
        f"Decision: {decision}.",
        f"Selected {selected_profile}; failures={selection['selected_failures'] or 'none'}.",
        f"YTD return={selection['selected_ytd_return_pct']:.2f}% vs required best={required_best_return_pct:.2f}% and SOXLBacktest={reference.return_pct:.2f}%.",
        f"WF MaxDD={selection['selected_wf_max_dd_pct']} with threshold < {TARGET_MAX_DD:.2f}%.",
        "No actor retention or runtime promotion is allowed from this diagnostic card.",
        f"Next mechanism: {next_mechanism}.",
    ],
}
write_json(CARD / "selection.json", json_safe(selection))
write_json(CARD / "summary.json", json_safe(summary))

lines = [
    "# Conclusion",
    "",
    f"Decision: `{decision}`.",
    "",
    f"Selected profile: `{selected_profile}`.",
    f"Failures: `{selection['selected_failures'] or 'none'}`.",
    f"YTD return: `{selection['selected_ytd_return_pct']:.2f}%` versus required best `{required_best_return_pct:.2f}%` and SOXLBacktest `{reference.return_pct:.2f}%`.",
    f"YTD MaxDD: `{selection['selected_ytd_max_dd_pct']:.2f}%`.",
    "",
    "Walk-forward metrics:",
]
for window in ("wf_2020", "wf_2022", "wf_2023_2024"):
    lines.append(
        f"- {window}: return={selection['selected_wf_return_pct'][window]:.2f}%, MaxDD={selection['selected_wf_max_dd_pct'][window]:.2f}%"
    )
lines.extend([
    "",
    "Risk note: this card refreshes and screens a path target only; it does not train, retain, or promote an actor.",
    f"Next action: `{next_mechanism}`.",
])
(CARD / "conclusion.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


{
  "experiment": "quantbench_ensemble_current_walkforward_v1",
  "created_at": "2026-06-13T13:13:53.078582+00:00",
  "decision": "quantbench_current_walkforward_not_target_ready_no_promotion",
  "next_mechanism": "use_failure_columns_to_tune_bull_release_or_drawdown_budget_before_actor_training",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "quantbench_paper_soxl_v2",
    "model_retention_walkforward_gate_v1",
    "cap10_three_fold_checkpoint_metric_replay_v2",
    "soxlbacktest_teacher_risk_budget_feasibility_v1"
  ],
  "reference_baseline": "soxlbacktest_qlib_buysell_reference",
  "role": "diagnose_training_target",
  "changed_layer": [
    "model_selection",
    "training_target",
    "quantbench_path_ensemble"
  ],
  "target_max_dd_pct": 20.0,
  "best_return_ratio": 0.95,
  "required_best_return_pct": 213.14346182652025,
  "required_soxlbacktest_return_pct": 132.07797795135497,
  "safe_backtest_end": "2026-06-11",
  "latest